# 8j — Preliminary composable forecast (four ways)

Implements `inst/1a_preliminary_framework_plan.md` + the fixes/extensions in `inst/1c`,
`inst/1d`, `inst/1e`: a joint renewal / next-generation-matrix model driven by **age-pair
contact-degree distributions**, scored **four ways** — the 2×2 grid of {unweighted
**NegBin**, weighted **Hurdle-Weibull**} degree models × {**Mean**, **Neighbourhood**} NGM.

The contact **mean** is estimated with **structural reciprocity** (`log μ_{i→j} = r + log Nⱼ`)
and **spatial-GP smoothing** across the age-pair grid (separable RBF, shared length-scale;
inst/1e). Forecasts use the **contact-updated iterate** over **4 origins** × 4 horizons;
**WIS** is computed on a **log scale** and aggregated **by horizon** via R `scoringutils`.

Fitting uses **Pathfinder.jl** (parsimonious fit / init) and optionally **Turing NUTS** (`USE_NUTS`).
See the specs for modelling details and the remaining lean simplifications (constant
within-window contacts; reduced transmission block).

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

USE_NUTS = false   # true ⇒ formal Turing NUTS fit (slow); false ⇒ Pathfinder parsimonious fit

## §1 Window, infection/antibody data, and age-pair degree data

In [ ]:
cfg  = FrameworkConfig()
grid = cis_age_grid()

# Four forecast origins ("forecasting sets", inst/1e). WIS is aggregated BY HORIZON
# across these origins (not per set). Each origin runs the contact-updated 4-week iterate.
FORECAST_ORIGINS = Date.(2021, 1, [3, 10, 17, 24])
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("forecast origins   : ", [w.origin for w in wins])
let w = wins[1], wd0 = load_window_data(wins[1]; grid = grid)
    println("origin[1] week     : ", w.origin)
    println("fit weeks          : ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("forecast targets   : ", w.forecast_weeks)
    println("weekly infections @ origin (age): ", round.(wd0.I_mean[:, end]; digits = 0))
    println("antibody @ origin (age)         : ", round.(wd0.antibody[:, end]; digits = 3))
end

In [ ]:
# Per-origin, per-horizon contact/degree windows (inst/1d iterate), shared across the
# four combos so each (origin, horizon) is binned once (not 4×). For horizon h the
# contact window slides to end at origin+h−1; infections & antibody stay frozen at the
# origin (rebuilt per origin inside the fit loop below).
apd_by_h_all = [
    [prepare_degree_data(
         WeeklyWindow(o + Day(7 * (h - 1));
                      n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
         cfg; grid = grid, setting = :all)
     for h in cfg.horizons]
    for o in FORECAST_ORIGINS]
println("prepared degree windows: ", length(apd_by_h_all), " origins × ",
        length(cfg.horizons), " horizons")

## §2 Fit the four combinations over four origins, forecast 1–4 weeks ahead

Each combo is a full joint fit (contact-degree + infection likelihoods). The contact **mean**
is a **reciprocity-structural, GP-smoothed** field: one symmetric log-rate per unordered age
pair, smoothed by a separable RBF over age-pair coordinates (midpoints, 70+→74.5; shared
length-scale), with `log μ_{i→j} = r + log Nⱼ` enforcing `Nᵢ·μ_{i→j} = Nⱼ·μ_{j→i}` exactly.

Forecasting is **contact-updated iterate** (inst/1d): from each origin t₀, for horizon
`h = 1..4` the model is **re-fit** with the contact/degree window slid to end at `t₀+h−1`
(infections & antibody frozen at t₀), the NGM refreshed, and one renewal step taken. The
neighbourhood-degree NGM uses the size-biased degree conditioned on non-zero contacts.
This runs for **4 origins** (2021-01-03 … 2021-01-24); MCMC chains are cached per
(origin, horizon) under `../dt_intermediate/8j_chn_*.jld2` — a re-run reloads them.

In [ ]:
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]

qtabs     = DataFrame[]
fc_store  = Dict{Tuple{Date,String},Array{Float64,3}}()
crps_rows = NamedTuple[]
for (oi, win_o) in enumerate(wins)
    wd_o     = load_window_data(win_o; grid = grid)
    truth_o  = load_forecast_truth(win_o; grid = grid)
    apd_by_h = apd_by_h_all[oi]
    for (dm, nb) in combos
        lbl = string(degree_label(dm), "|", ngm_label(nb))
        println("\n--- origin ", win_o.origin, "  ", lbl,
                "  (contact-updated iterate, USE_NUTS=", USE_NUTS, ") ---")
        # re-fits per horizon (contact window → origin+h−1, infections frozen at origin);
        # chains cached under ../dt_intermediate/8j_chn_<…>_<origin>_h<h>.jld2 (reloaded on re-run).
        fc = iterated_forecast(dm, nb, wd_o, cfg, win_o;
                               grid = grid, setting = :all, use_nuts = USE_NUTS,
                               save_dir = "../dt_intermediate", apd_by_h = apd_by_h)
        fc_store[(win_o.origin, lbl)] = fc
        push!(qtabs, to_quantile_long(fc, truth_o, lbl, win_o, cfg, grid.LAB))
        push!(crps_rows, (origin = win_o.origin, model = lbl,
                          mean_crps = mean_crps(fc, truth_o)))
    end
end
qall = vcat(qtabs...)              # all origins × combos, tagged by forecast_date=origin
println("\nquantile rows: ", size(qall), "  (", length(wins), " origins × ",
        length(combos), " combos)")
size(qall)

## §3 Score four ways — WIS via `scoringutils`

In [ ]:
scores = score_wis(qall)   # scores both natural & log scale; aggregated by horizon (inst/1e)

# Headline: LOG-SCALE WIS aggregated BY HORIZON across the four origins.
by_mh_log = sort(@subset(scores.by_model_h, :scale .== "log"), [:model, :horizon])
by_m_log  = sort(@subset(scores.by_model,   :scale .== "log"), :wis)

println("\n===== log-scale WIS by model (aggregated over horizons & origins) =====")
show(by_m_log, allcols = true); println()
println("\n===== log-scale WIS by model × horizon (aggregated over origins) =====")
show(by_mh_log, allcols = true); println()

# native sample-CRPS cross-check, averaged over the four origins
crps_df = sort(combine(groupby(DataFrame(crps_rows), :model), :mean_crps => mean => :mean_crps),
               :mean_crps)
println("\nmean native CRPS (avg over origins):")
show(crps_df, allrows = true); println()

CSV.write("../res/8j_scores_by_model.csv", scores.by_model)          # both scales
CSV.write("../res/8j_scores_by_model_horizon.csv", scores.by_model_h) # both scales × horizon
by_mh_log

In [ ]:
# log-scale WIS by horizon (one line per model), aggregated over the four origins
Hn = length(cfg.horizons)
wis_h_fig = plot(; xlabel = "horizon (weeks)", ylabel = "mean log-scale WIS",
                 title = "8j — log-scale WIS by horizon (4 origins aggregated)",
                 size = (760, 420), legend = :topleft, xticks = 1:Hn)
for m in unique(by_mh_log.model)
    sub = sort(@subset(by_mh_log, :model .== m), :horizon)
    plot!(wis_h_fig, sub.horizon, sub.wis; marker = :circle, lw = 2, label = m)
end
savefig(wis_h_fig, "../res/8j_wis_by_horizon.png")

# and the aggregated four-ways bar (log scale)
wis_bar = bar(by_m_log.model, by_m_log.wis; orientation = :h, legend = false,
              xlabel = "mean log-scale WIS (lower = better)",
              title = "8j — log-scale WIS four ways (aggregated)",
              size = (760, 360), left_margin = 8Plots.mm)
savefig(wis_bar, "../res/8j_wis_four_ways.png")
wis_h_fig

In [ ]:
# Forecast fan (total infections across ages) vs realized — representative origin
origin_show = FORECAST_ORIGINS[1]
truth_show  = load_forecast_truth(wins[1]; grid = grid)
qs_lo, qs_hi = 0.05, 0.95
H = length(cfg.horizons)
panels = Plots.Plot[]
for (dm, nb) in combos
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    fc  = fc_store[(origin_show, lbl)]
    tot = dropdims(sum(fc; dims = 1); dims = 1)          # H × draws
    med = [median(tot[h, :]) for h in 1:H]
    lo  = [quantile(tot[h, :], qs_lo) for h in 1:H]
    hi  = [quantile(tot[h, :], qs_hi) for h in 1:H]
    tr  = [sum(truth_show[:, h]) for h in 1:H]
    p = plot(1:H, med; ribbon = (med .- lo, hi .- med), lw = 2, label = "forecast",
             title = lbl, titlefontsize = 7, xlabel = "horizon (wk)", legend = false)
    plot!(p, 1:H, tr; lw = 2, color = :black, marker = :circle, label = "observed")
    push!(panels, p)
end
fan = plot(panels...; layout = (2, 2), size = (900, 640),
           plot_title = "8j — total infection forecast vs observed (origin $origin_show, 90% band)",
           plot_titlefontsize = 10)
savefig(fan, "../res/8j_forecast_fan.png")
fan

## §4 Notes

- **Four ways** = the 2×2 grid; the headline metric is **log-scale WIS aggregated by horizon**
  across the four origins (`res/8j_scores_by_model_horizon.csv`, `scale=="log"`), with
  over/under-prediction & dispersion components, bias, and 50/90% coverage.
- **Reciprocity + GP smoothing of the mean** (inst/1e): the contact mean is a *symmetric*
  log-rate over the 28 unordered age pairs, `log μ_{i→j} = r_{min,max} + log(popⱼ)` (so
  `popᵢ·μ_{i→j} = popⱼ·μ_{j→i}` exactly), with `r` a **separable-RBF GP** over age midpoints
  (70+ → 74.5; shared length-scale ρ; non-centred `f = η·L·z`). This replaces the 49 iid
  per-cell deviations and the post-hoc mean balancing; the neighbourhood NGM is still
  reciprocity-balanced (size-biased C0 is not reciprocal even when μ is).
- **Group-contact duration weight** (inst/1e): `:cnt_mass=="mass"` contacts (no recorded
  duration, ~half of contact rows) get the explicit `cfg.w_dur_group = 2.5/240` (fixed now,
  estimable later) instead of the `<5 min` bin; individually-reported contacts use the
  duration bins. Counts (NegBin path) are unchanged.
- **Neighbourhood-degree NGM among non-zero** (inst/1c, 1d): `C0 = ⟨k²⟩/⟨k⟩ × g`,
  `g = 1/(1−P₀)` (NegBin, floored) / `g = (1−p⁰)` (hurdle-Weibull). Mean NGM (`C0 = ⟨k⟩`) unchanged.
- **Contact-updated iterate over 4 origins** (inst/1d, 1e): infections/antibody frozen at each
  origin; the contact matrix is re-estimated each horizon (window ending origin+h−1); renewal
  stepped forward one week at a time (mean-plugged lags). Chains cached per (origin, horizon).
- **WIS on a log scale** (inst/1e): `transform_forecasts(fun=log_shift, offset=1)` in
  `score_wis`; both natural and log scores are written out, log is the headline.
- The run uses **Pathfinder** draws by default (`USE_NUTS=false`); set `USE_NUTS=true` for the
  formal Turing NUTS fit (much slower). Remaining lean simplifications: within-window contacts
  constant per cell (temporal GP is the swap-in seam); reference transmission block; 5-day
  generation interval. Revisit before scientific interpretation.